In [ ]:
import dotenv

import rasterio
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterstats

from rasterstats import zonal_stats
from rasterio.warp import reproject, Resampling, calculate_default_transform
from rasterio.mask import mask

from food_security import salinity_correction, water_quality
from food_security.fao_api import FAOClient

from pathlib import Path

In [ ]:
from pathlib import Path
import dotenv

print("Current folder:", Path.cwd())
print(".env location:", Path(".env").resolve())
print(".env exists:", Path(".env").exists())

config = dotenv.dotenv_values(".env")
print("Variables found:", config.keys())

In [ ]:
config = dotenv.dotenv_values(".env")
username = config["FAOSTAT_USERNAME"]
password = config["FAOSTAT_PASSWORD"]

fao_client = FAOClient(username=username, password=password)

In [ ]:
src_dir = Path('~').expanduser() / "OneDrive - Stichting Deltares/PhD/Egypt_ERF_data/ribasim/calibration_capmas/"


In [ ]:
from egypt_survey_ml.production import bind_salinity_correction, get_hectares_fixed
from egypt_survey_ml.io_utils import write_excel_file
bind_salinity_correction()


In [ ]:
toml_file = src_dir.parent / "salinity_correction_egypt_capmas.toml"
corrected_df = salinity_correction.generate_crop_yield_csv(
    config_path=toml_file,
    add_labor=False,
    save=False,
    fao_client=fao_client
)

In [ ]:
print (corrected_df)

In [ ]:
relevant_crops = [
    "Bean_FBEAN1 (ha)",
    "LongBerseem_LBSEEM1 (ha)",
    "ShortBerseem_SBSEMW1 (ha)",
    "Cotton_SDELS1 (ha)",
    "SummerMaize_MAIZES1 (ha)",
    "Orchard_CITRUS1 (ha)",
    "LongRice_PADDY1 (ha)",
    "SumerSorghum_SORGMS1 (ha)",
    "SugarBeet_SBEET1 (ha)",
    "SugarCane_SCANE1 (ha)",
    "SumerVegtabl_VEGETS1 (ha)",
    "Wheat_WHEAT1 (ha)",
    "WintrVegtabl_VEGETW1 (ha)",
    "NiliVegetabl_VEGETN1 (ha)",
    "NiliMaize_MAIZEN1 (ha)",
    "Barley_BARLEY1 (ha)",
    "NiliTomato_TMATON1 (ha)",
    "NiliPotato_PTATON1 (ha)",
    "NiliSorghum_SORGMN1 (ha)",
    "SummerTomato_TMATOS1 (ha)",
    "SummerPotato_PTATOS1 (ha)",
    "SummerOnion_ONIONS1 (ha)",
    "WinterTomato_TMATOW1 (ha)",
    "WinterOnion_ONIONW1 (ha)",
    "OtherLegume_OLGUME1 (ha)",
    "Flax_FLAX1 (ha)",
    "Soybean_SBEAN1 (ha)",
    "Lentil_LENTIL1 (ha)",
    "Sesame_SESAME1 (ha)",
    "Peanut_GNUT1 (ha)",
    "WinterPotato_PTATOW1 (ha)"
]

excel_salinity_df = pd.DataFrame(
    {
        "crop_name": corrected_df["crop_name"],
        "crop_name_fao": corrected_df["crop_name_fao"],
        "area": corrected_df["area_map_name"],
        "yield": corrected_df["corrected_yield"],
        "cultivation_area (ha)": corrected_df["hectares"],
        "year": corrected_df["year"],
    }
)
    
excel_salinity_df = excel_salinity_df[excel_salinity_df['crop_name'].isin(relevant_crops)]

In [ ]:
excel_salinity_df.to_csv("a.csv")

In [ ]:
from egypt_survey_ml.io_utils import write_excel_file


In [ ]:
command_gdf = gpd.read_file(src_dir / 'Final2_Command_Area.shp')
conversion_df = pd.read_excel(excel_path, sheet_name='command_area')

mapping = (
    command_gdf.merge(
        conversion_df[['area_map_name', 'area_name']],
        left_on='OBJECTID',
        right_on='area_map_name'
    )
    .set_index('Name')['area_name']
)

excel_salinity_df['area'] = excel_salinity_df['area'].map(mapping)

In [ ]:
write_excel_file(excel_salinity_df, excel_path, sheet_name="production_salinity")

In [ ]:
pp_df = pd.read_excel(excel_path, sheet_name='farm_gate_price')

excel_crop_df = pd.DataFrame(
    {
        "area": excel_salinity_df["area"].unique()
    }
)

for i, row in excel_crop_df.iterrows():
    pp_total = 0
    area_name = row["area"]
    salinity_crops_row = excel_salinity_df[(excel_salinity_df['area'] == area_name) & (excel_salinity_df['year'] == 2021)]
    for crop_name in relevant_crops:
        salinity_crop_row = salinity_crops_row[salinity_crops_row['crop_name'] == crop_name]
        crop_name_fao = salinity_crop_row['crop_name_fao'].iloc[0]
        excel_crop_df.loc[i, f"salinity_production_{crop_name_fao}"] = salinity_crop_row['yield'].iloc[0]
        excel_crop_df.loc[i, f"salinity_cultivation_area_{crop_name_fao}"] = salinity_crop_row['cultivation_area (ha)'].iloc[0]

        pp_row = pp_df[pp_df['crop_name_fao'] == crop_name_fao]
        pp_crop = salinity_crop_row['yield'].iloc[0] * pp_row['Farmgate price\n(000 EGP/ton)'].iloc[0]
        pp_total += pp_crop
    
    excel_crop_df.loc[i, 'producer_price'] = pp_total

In [ ]:
excel_command_df = pd.read_excel(excel_path, sheet_name='command_unit')

excel_command_df = (
    excel_command_df
    .drop(columns=excel_crop_df.columns.difference(["area"]), errors="ignore")
    .merge(excel_crop_df, on="area", how="left")
)

for column in excel_crop_df.columns:
    if column != "area" and column != 'producer_price' and 'cultivation' not in column:
        excel_command_df[column] = excel_command_df[column] / excel_command_df['rural_population']
    if column == "producer_price":
        excel_command_df[column] = excel_command_df[column] / excel_command_df['arable_km2']

write_excel_file(excel_command_df, excel_path, sheet_name='command_unit')

In [ ]:
excel_command_df